In [1]:
import os
import mne
import numpy as np
import pandas as pd
from scipy import interpolate
import matplotlib.pyplot as plt
import scipy
import sklearn
from scipy.signal import resample
import json
import warnings
from pathlib import Path
from collections import defaultdict
warnings.filterwarnings("ignore")

In [2]:
import mne
from mne.preprocessing import ICA
try:
    from mne_icalabel import label_components
except Exception:
    label_components = None

In [3]:
# multi-scale sampling rates
SAMPLE_RATE_LIST = [200, 100, 50]  # Hz

# fixed segment length in samples (timestamps)
SEG_LEN = 400  # number of time steps per segment
# overlap in samples (timestamps), not in seconds
OVERLAP = 200    # e.g. 200 means 50% overlap for SEG_LEN=400

assert 0 <= OVERLAP < SEG_LEN, "OVERLAP must be in [0, SEG_LEN)."

sub_folder_path = f"L{SEG_LEN}"
sub_folder_path

'L400'

In [4]:
root = 'FEPCR/'

In [5]:
# root dir
root_1 = 'FEPCR/FEPCR-1/'
# participants file path
participants_path_1 = os.path.join(root_1, 'participants.tsv')
participants_1 = pd.read_csv(participants_path_1, sep='\t')
participants_1

,participant_id,type,age,gender,race,ethnicity
0,sub-1448,Control,15.734428,M,White,Not Hispanic
1,sub-1824,Psychosis,21.497604,F,White,Not Hispanic
2,sub-1971,Control,22.442163,F,Black,Not Hispanic
3,sub-1983,Psychosis,19.351129,M,White,Not Hispanic
4,sub-1989,Control,22.061602,F,White,Not Hispanic
...,...,...,...,...,...,...
77,sub-2184A,Psychosis,23.444216,M,Black,Not Hispanic
78,sub-2193A,Control,19.739904,M,White,Not Hispanic
79,sub-2214A,Control,29.081451,M,White,Not Hispanic
80,sub-2217A,Psychosis,20.366872,M,White,Not Hispanic


In [6]:
# root dir
root_2 = 'FEPCR/FEPCR-2/'
# participants file path
participants_path_2 = os.path.join(root_2, 'participants.tsv')
participants_2 = pd.read_csv(participants_path_2, sep='\t')
participants_2

,participant_id,type,age,gender,race,ethnicity
0,sub-2235A,Control,22.023272,M,Asian,Not Hispanic
1,sub-2237A,Psychosis,18.184805,F,Black,Not Hispanic
2,sub-2238A,Control,21.040383,M,White,Not Hispanic
3,sub-2240A,Control,38.258727,F,Black,Not Hispanic
4,sub-2245A,Psychosis,27.071869,M,White,Not Hispanic
...,...,...,...,...,...,...
56,sub-2530A,Psychosis,15.906913,M,White,Not Hispanic
57,sub-2538A,Control,27.184120,F,Asian,Hispanic
58,sub-2574A,Control,23.739904,F,White,Not Hispanic
59,sub-2578A,Psychosis,18.773443,M,Asian,Not Hispanic


In [7]:
participants = pd.concat([participants_1, participants_2], ignore_index=True)
participants

,participant_id,type,age,gender,race,ethnicity
0,sub-1448,Control,15.734428,M,White,Not Hispanic
1,sub-1824,Psychosis,21.497604,F,White,Not Hispanic
2,sub-1971,Control,22.442163,F,Black,Not Hispanic
3,sub-1983,Psychosis,19.351129,M,White,Not Hispanic
4,sub-1989,Control,22.061602,F,White,Not Hispanic
...,...,...,...,...,...,...
138,sub-2530A,Psychosis,15.906913,M,White,Not Hispanic
139,sub-2538A,Control,27.184120,F,Asian,Hispanic
140,sub-2574A,Control,23.739904,F,White,Not Hispanic
141,sub-2578A,Psychosis,18.773443,M,Asian,Not Hispanic


In [8]:
# label mapping: diagnosis code -> label id
label_map = {'Control': 0, 'Psychosis': 1}

# build subject info: "sub-XXX" -> (label, pid)
sub_info = {}  # key: subject name, value: (label, pid)
for row in participants.itertuples(index=False):
    sub_name = row[0]          # e.g. 'sub-001'
    diag_code = row[1]         # e.g. 'A' / 'F' / 'C'
    label = label_map[diag_code]
    sub_info[sub_name] = label

len(sub_info), list(sub_info.items())[:5]

(143,
 [('sub-1448', 0),
  ('sub-1824', 1),
  ('sub-1971', 0),
  ('sub-1983', 1),
  ('sub-1989', 0)])

## Features

In [9]:
def ensure_minimal_vmrk(vhdr_path: str):
    vhdr = Path(vhdr_path)
    vmrk = vhdr.with_suffix('.vmrk')
    if not vmrk.exists():
        vmrk.write_text(
            "Brain Vision Data Exchange Marker File, Version 1.0\n"
            "; Auto-created to allow reading continuous data\n"
            "[Common Infos]\n"
            "Codepage=UTF-8\n\n"
            "[Marker Infos]\n"
            "; Mk<NR>=<Type>,<Description>,<Position>,<Size>,<Channel>\n"
            "Mk1=New Segment,,1,1,0\n",
            encoding="utf-8"
        )

    # 确保 .vhdr 里存在 MarkerFile= 并指向文件名
    txt = vhdr.read_text(encoding="utf-8", errors="ignore")
    if "[Common Infos]" not in txt or "MarkerFile=" not in txt.split("[Common Infos]")[1]:
        txt = txt.replace("[Common Infos]", f"[Common Infos]\nMarkerFile={vmrk.name}", 1)
        vhdr.write_text(txt, encoding="utf-8")

In [10]:
# Test for bad channels, sampling freq and shape
"""bad_channel_list, sampling_freq_list, data_shape_list = [], [], []
for task in os.listdir(root):
    task_path = os.path.join(root, task)
    for sub in os.listdir(task_path):
        if 'sub-' in sub:
            sub_path = os.path.join(task_path, sub, 'eeg/')
            # print(sub_path)
            for file in os.listdir(sub_path):
                if '.vhdr' in file:
                    file_path = os.path.join(sub_path, file)
                    ensure_minimal_vmrk(file_path)
                    raw = mne.io.read_raw_brainvision(file_path, preload=True)
                    # print(raw.get_montage())
                    # get bad channels
                    bad_channel = raw.info['bads']
                    bad_channel_list.append(bad_channel)
                    # get sampling frequency
                    sampling_freq = raw.info['sfreq']
                    sampling_freq_list.append(sampling_freq)
                    # get eeg data
                    data = raw.get_data()
                    data_shape = data.shape
                    print(data_shape, bad_channel, sampling_freq, file_path)
                    data_shape_list.append(data_shape)"""

"bad_channel_list, sampling_freq_list, data_shape_list = [], [], []\nfor task in os.listdir(root):\n    task_path = os.path.join(root, task)\n    for sub in os.listdir(task_path):\n        if 'sub-' in sub:\n            sub_path = os.path.join(task_path, sub, 'eeg/')\n            # print(sub_path)\n            for file in os.listdir(sub_path):\n                if '.vhdr' in file:\n                    file_path = os.path.join(sub_path, file)\n                    ensure_minimal_vmrk(file_path)\n                    raw = mne.io.read_raw_brainvision(file_path, preload=True)\n                    # print(raw.get_montage())\n                    # get bad channels\n                    bad_channel = raw.info['bads']\n                    bad_channel_list.append(bad_channel)\n                    # get sampling frequency\n                    sampling_freq = raw.info['sfreq']\n                    sampling_freq_list.append(sampling_freq)\n                    # get eeg data\n                    data 

In [11]:
"""from collections import Counter

print(bad_channel_list)
print("Channel number counter:", Counter(i[0] for i in data_shape_list))
print("Sampling rate counter:", Counter(sampling_freq_list))"""

'from collections import Counter\n\nprint(bad_channel_list)\nprint("Channel number counter:", Counter(i[0] for i in data_shape_list))\nprint("Sampling rate counter:", Counter(sampling_freq_list))'

In [12]:
def data_preprocessing(
    raw: mne.io.Raw,
    notch_freq: float = 60.0,
    l_freq: float = 0.5,
    h_freq: float = 40.0,
    do_bad_interp: bool = True,
    verbose: bool = True,
):
    """
    Preprocessing steps ：
      2) Set Montage 
      3) 60 Hz Notch（before band pass）
      4) bandpass filter（default 0.5–40 Hz）
      5) interpolate bad channels（if do_bad_interp is True）
      6) re-reference to average
      7) ICA（在 1 Hz 高通的副本上拟合，自动剔除眼动/肌电等分量，需 mne-icalabel）
      8) downsample to 250 Hz
    """
    
    # 1. Remove unknown channel
    unknown_channel = ['O9', 'O10']
    channel_list = list(raw.ch_names)
    new_channel_list = list(set(channel_list) - set(unknown_channel))
    raw.pick(new_channel_list)
        
    # 2. Set Montage
    raw.set_montage(mne.channels.make_standard_montage('standard_1005'))
    if verbose:
        print("✔ Step 2, Montage set: 'standard_1005'.")
        
    # 3. Notch（工频）
    if notch_freq is not None:
        raw.notch_filter(freqs=[notch_freq], picks="eeg", verbose=False)
        if verbose:
            print(f"✔ Step 3: Notch @ {notch_freq} Hz")
        
    # 4. Bandpass Filter (0.5–40 Hz)
    raw.filter(l_freq=l_freq, h_freq=h_freq, picks="eeg", verbose=False)
    if verbose:
        print(f"✔ Step 4: Band-pass {l_freq}–{h_freq} Hz")
        
    # 5. Interpolate bad channels
    if do_bad_interp and raw.info.get("bads"):
        raw.interpolate_bads(reset_bads=True, verbose=False)
        if verbose:
            print(f"✔ Step 5: Interpolated bads: {raw.info.get('bads', [])}")
    else:
        if verbose:
            print("ℹ Step 5: No bads to interpolate (set raw.info['bads'] first if needed)")
            
    # 6) Re-reference to average
    raw.set_eeg_reference("average", verbose=False)
    if verbose:
        print("✔ Step 6: Average reference")
    
    # 5. ICA + ICLabel
    raw_ica = raw.copy()
    ica = ICA(n_components=None, random_state=97, max_iter='auto')
    ica.fit(raw_ica)
    if verbose:
        print("✔ ICA fitted.")

    try:
        ic_labels = label_components(raw_ica, ica, method='iclabel')
        labels = ic_labels['labels']
        probs = ic_labels['y_pred_proba']

        artifact_thresholds = {
            'eye blink': 0.7,
            'muscle artifact': 0.6,
            'heart beat': 0.5,
            'line noise': 0.8,
            'channel noise': 0.9
        }

        to_exclude = [
            i for i, label in enumerate(labels)
            if label in artifact_thresholds and probs[i] >= artifact_thresholds[label]
        ]
        if to_exclude:
            ica.exclude = to_exclude
            ica.apply(raw)
            if verbose:
                print(f"✔ ICA applied. Excluded components: {to_exclude}")
        else:
            if verbose:
                print("No ICs exceeded artifact thresholds. No components excluded.")

    except Exception as e:
        if verbose:
            print(f"⚠ ICLabel failed: {e}. Proceeding without ICA-based removal.")
        
    return raw

In [13]:
def resample_time_series(data, original_fs, target_fs):
    """
    Resample each channel independently.
    data: (T_raw, C)
    return: (T_new, C)
    """
    T, C = data.shape
    new_length = int(T * target_fs / original_fs)
    resampled_data = np.zeros((new_length, C), dtype=np.float32)
    for i in range(C):
        # resample one channel
        resampled_data[:, i] = resample(data[:, i], new_length)
    return resampled_data


def compute_step(seg_len, overlap):
    """
    Compute sliding step (in samples) given segment length and overlap.
    """
    step = seg_len - overlap
    if step <= 0:
        raise ValueError(f"Invalid overlap={overlap}: step <= 0.")
    return step


def compute_num_segments(num_samples, seg_len, step):
    """
    Compute how many segments can be extracted from a sequence
    with given segment length and step.
    """
    if num_samples < seg_len:
        return 0
    return 1 + (num_samples - seg_len) // step

## PASS 1: Find total number of segments across all subjects

In [14]:
# Loop through each subject and session to preprocess the EEG data
session_segment_counts = defaultdict(lambda: defaultdict(int))
N_total = 0
# 19 standard channels should be: 'Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T3', 'C3', 'Cz', 'C4', 'T4', 'T5', 'P3', 'Pz', 'P4', 'T6', 'O1', 'O2'
# For here, T7, T8 is the same to T3, T4; P7, P8 is the same to T5, T6
# So we use T7, T8, P7, P8 to replace T3, T4, T5, T6
standard_channels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 
                     'Cz', 'C4', 'T8', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'O2']
n_channels = len(standard_channels)

# step (timestamps)
STEP = compute_step(SEG_LEN, OVERLAP)
print("SEG_LEN =", SEG_LEN, "OVERLAP =", OVERLAP, "STEP =", STEP, "\n")
    
sub_id = 1
for task in os.listdir(root):
    task_path = os.path.join(root, task)
    for sub in os.listdir(task_path):
        if 'sub-' in sub:
            sub_path = os.path.join(task_path, sub, 'eeg/')
            # print(sub_path)
            for file in os.listdir(sub_path):
                if '.vhdr' in file:
                    file_path = os.path.join(sub_path, file)
                    channal_tsv_path = os.path.join(sub_path, file.replace('eeg.vhdr', 'channels.tsv'))
                    channels = pd.read_csv(channal_tsv_path, sep='\t')
                    channel_list = channels['name'].tolist()
                    # No channel information in the file, so we use the channel names from the channels.tsv file and create a new Raw object
                    raw = mne.io.read_raw_brainvision(file_path, preload=True)
                    source_sample_rate = raw.info['sfreq']
                    data = raw.get_data()
                    info = mne.create_info(ch_names=channel_list, sfreq=source_sample_rate, ch_types=['eeg'] * len(channel_list))
                    raw = mne.io.RawArray(data, info)
                    print("Original Channels: ", raw.ch_names)
                    # For here, T7, T8 is close to T3, T4; P7, P8 is the same to T5, T6;
                    # So we use T7, T8, P7, P8 to replace T3, T4, T5, T6
                    standard_channels = []
                    if task == 'FEPCR-1':
                        # In task 1, some channels are capital, e.g, FP1, FP2, etc.
                        standard_channels = ['FP1', 'FP2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 
                                         'Cz', 'C4', 'T8', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'O2']  
                    elif task == 'FEPCR-2':
                        standard_channels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 
                                         'Cz', 'C4', 'T8', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'O2']
                    raw.pick(standard_channels)
                    lower_standard_channels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 
                                         'Cz', 'C4', 'T8', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'O2']  # lowercase to match MNE, e.g, Fp1, Fp2, etc.
                    data = raw.get_data()
                    info = mne.create_info(ch_names=lower_standard_channels, sfreq=source_sample_rate, ch_types=['eeg'] * len(lower_standard_channels))
                    raw = mne.io.RawArray(data, info)
                    T_raw = raw.n_times
                    for fs in SAMPLE_RATE_LIST:
                        T_res = int(T_raw * fs / source_sample_rate)
                        # compute number of segments
                        n_seg = compute_num_segments(T_res, SEG_LEN, STEP)
                        session_segment_counts[sub_id][fs] += n_seg
                        N_total += n_seg
                        print(f"fs={fs}Hz: T_res={T_res}, STEP={STEP}, n_seg={n_seg}")
                        print("----------------------------------------")
            sub_id += 1
        print("-------------------------------------\n")

print("\n===================================")
print("Total segments N_total =", N_total)
print("Channels =", n_channels)
print("===================================")

if N_total == 0:
    raise RuntimeError("No segments found. Please check SEG_LEN / OVERLAP.")

SEG_LEN = 400 OVERLAP = 200 STEP = 200 

-------------------------------------

-------------------------------------

-------------------------------------

-------------------------------------

-------------------------------------

-------------------------------------

Extracting parameters from FEPCR/FEPCR-1\sub-1448\eeg/sub-1448_task-Rest_eeg.vhdr...
Setting channel info structure...
Reading 0 ... 305999  =      0.000 ...   305.999 secs...
Creating RawArray with float64 data, n_channels=64, n_times=306000
    Range : 0 ... 305999 =      0.000 ...   305.999 secs
Ready.
Original Channels:  ['FP1', 'FPz', 'FP2', 'AF7', 'AF3', 'AF4', 'AF6', 'F7', 'F5', 'F3', 'F1', 'Fz', 'F2', 'F4', 'F6', 'F8', 'FT9', 'FT7', 'FC5', 'FC1', 'FC2', 'FC6', 'FT8', 'FT10', 'T9', 'T7', 'C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6', 'T8', 'T10', 'TP9', 'TP7', 'CP3', 'CP1', 'CP2', 'CP4', 'TP8', 'TP10', 'P7', 'P5', 'P3', 'P1', 'Pz', 'P2', 'P4', 'P6', 'P8', 'PO7', 'PO3', 'PO4', 'PO8', 'O1', 'Oz', 'O2', 'Iz', 'VEOG',

In [19]:
output_root = os.path.join('Processed', sub_folder_path, 'FEPCR')
os.makedirs(output_root, exist_ok=True)

X_path = os.path.join(output_root, 'X.dat')
y_path = os.path.join(output_root, 'y.dat')
meta_path = os.path.join(output_root, 'meta.json')

print("X path:", X_path)
print("y path:", y_path)

# create memmap storage
X_mm = np.memmap(X_path, dtype='float32', mode='w+', shape=(N_total, SEG_LEN, n_channels))
y_mm = np.memmap(y_path, dtype='float32', mode='w+', shape=(N_total, 3))

X path: Processed\L400\FEPCR\X.dat
y path: Processed\L400\FEPCR\y.dat


## PASS 2: Process and store segments into memmap

In [20]:
cur = 0  # current index in memmap
total_seconds_all = 0  # used for total hours calculation
    
sub_id = 1
for task in os.listdir(root):
    task_path = os.path.join(root, task)
    for sub in os.listdir(task_path):
        if 'sub-' in sub:
            sub_path = os.path.join(task_path, sub, 'eeg/')
            # print(sub_path)
            sub_label = sub_info[sub]
            for file in os.listdir(sub_path):
                if '.vhdr' in file:
                    file_path = os.path.join(sub_path, file)
                    channal_tsv_path = os.path.join(sub_path, file.replace('eeg.vhdr', 'channels.tsv'))
                    channels = pd.read_csv(channal_tsv_path, sep='\t')
                    channel_list = channels['name'].tolist()
                    # No channel information in the file, so we use the channel names from the channels.tsv file and create a new Raw object
                    raw = mne.io.read_raw_brainvision(file_path, preload=True)
                    source_sample_rate = raw.info['sfreq']
                    data = raw.get_data()
                    info = mne.create_info(ch_names=channel_list, sfreq=source_sample_rate, ch_types=['eeg'] * len(channel_list))
                    raw = mne.io.RawArray(data, info)
                    print("Original Channels: ", raw.ch_names)
                    # For here, T7, T8 is close to T3, T4; P7, P8 is the same to T5, T6;
                    # So we use T7, T8, P7, P8 to replace T3, T4, T5, T6
                    standard_channels = []
                    if task == 'FEPCR-1':
                        # In task 1, some channels are capital, e.g, FP1, FP2, etc.
                        standard_channels = ['FP1', 'FP2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 
                                         'Cz', 'C4', 'T8', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'O2']  
                    elif task == 'FEPCR-2':
                        standard_channels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 
                                         'Cz', 'C4', 'T8', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'O2']
                    raw.pick(standard_channels)
                    lower_standard_channels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 
                                         'Cz', 'C4', 'T8', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'O2']  # lowercase to match MNE, e.g, Fp1, Fp2, etc.
                    data = raw.get_data()
                    info = mne.create_info(ch_names=lower_standard_channels, sfreq=source_sample_rate, ch_types=['eeg'] * len(lower_standard_channels))
                    raw = mne.io.RawArray(data, info)
                    raw = data_preprocessing(
                        raw=raw,
                        notch_freq=60,  # data collected in United States, so notch at 60 Hz
                        l_freq=0.5,
                        h_freq=45.0,
                        do_bad_interp=True,
                        verbose=True
                    )
                    raw.reorder_channels(lower_standard_channels)
                    data = raw.get_data().T  # (T_raw, C)
                    T_raw = raw.n_times
                    total_seconds_all += data.shape[0] / source_sample_rate
                    for fs in SAMPLE_RATE_LIST:
                        # resample to target fs
                        data_resampled = resample_time_series(data, source_sample_rate, fs)
                        T_res, _ = data_resampled.shape
                        print(f"fs={fs}Hz: resampled shape={data_resampled.shape}")
            
                        # overlapped sliding window with fixed STEP (in timestamps)
                        starts = np.arange(0, T_res - SEG_LEN + 1, STEP, dtype=int)
                        print(f"fs={fs}Hz: segments={len(starts)}")
            
                        for s in starts:
                            if cur >= N_total:
                                raise RuntimeError("Exceeded predicted N_total.")
            
                            X_mm[cur] = data_resampled[s:s + SEG_LEN]   # (SEG_LEN, C)
                            y_mm[cur, 0] = float(sub_label)       # label
                            y_mm[cur, 1] = float(sub_id)      # Global subject ID
                            y_mm[cur, 2] = float(fs)          # sample rate (scale)
                            cur += 1
                    sub_id += 1
        print("-------------------------------------\n")

total_hours_all = total_seconds_all / 3600.0
print("DONE: cur =", cur, " expected N_total =", N_total)
print(f"Total hours across all subjects: {total_hours_all:.2f} hours")

# flush
del X_mm
del y_mm

# save meta
meta = {
    "N": int(N_total),
    "T": SEG_LEN,
    "C": int(n_channels),
    "SAMPLE_RATE_LIST": SAMPLE_RATE_LIST,
    "OVERLAP": int(OVERLAP),   # in samples
    "STEP": int(STEP),
    "X_path": X_path,
    "y_path": y_path,
}
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Saved meta:", meta_path)

-------------------------------------

-------------------------------------

-------------------------------------

-------------------------------------

-------------------------------------

-------------------------------------

Extracting parameters from FEPCR/FEPCR-1\sub-1448\eeg/sub-1448_task-Rest_eeg.vhdr...
Setting channel info structure...
Reading 0 ... 305999  =      0.000 ...   305.999 secs...
Creating RawArray with float64 data, n_channels=64, n_times=306000
    Range : 0 ... 305999 =      0.000 ...   305.999 secs
Ready.
Original Channels:  ['FP1', 'FPz', 'FP2', 'AF7', 'AF3', 'AF4', 'AF6', 'F7', 'F5', 'F3', 'F1', 'Fz', 'F2', 'F4', 'F6', 'F8', 'FT9', 'FT7', 'FC5', 'FC1', 'FC2', 'FC6', 'FT8', 'FT10', 'T9', 'T7', 'C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6', 'T8', 'T10', 'TP9', 'TP7', 'CP3', 'CP1', 'CP2', 'CP4', 'TP8', 'TP10', 'P7', 'P5', 'P3', 'P1', 'Pz', 'P2', 'P4', 'P6', 'P8', 'PO7', 'PO3', 'PO4', 'PO8', 'O1', 'Oz', 'O2', 'Iz', 'VEOG', 'Misc', 'ECG', 'M2']
Creating RawArray w

## Load and check the processed data

In [21]:
import json
import numpy as np

# load meta information
meta_path = meta_path  # if you already have meta_path in current notebook
with open(meta_path, "r") as f:
    meta = json.load(f)

N = meta["N"]
T = meta["T"]          # SEG_LEN
C = meta["C"]
X_path = meta["X_path"]
y_path = meta["y_path"]

print("Meta:")
print("  N =", N)
print("  T =", T)
print("  C =", C)
print("  X_path =", X_path)
print("  y_path =", y_path)
print("-------------------------------------")

# open memmap
X_mm = np.memmap(X_path, dtype='float32', mode='r', shape=(N, T, C))
y_mm = np.memmap(y_path, dtype='float32', mode='r', shape=(N, 3))

# subject_id is stored in y[:, 1]
subject_ids = np.unique(y_mm[:, 1]).astype(int)

for sid in sorted(subject_ids):
    # find all indices for this subject
    idx = np.where(y_mm[:, 1] == sid)[0]

    # compute shapes logically (do not load all X into memory)
    n_seg = len(idx)
    x_shape = (n_seg, T, C)    # logical X shape for this subject
    y_shape = (n_seg, 3)       # logical y shape for this subject

    # get sampling rates for all segments of this subject
    fs_subject = y_mm[idx, 2]  # shape (n_seg,)


    print(f"Subject ID {sid:03d}: X shape={x_shape}, y shape={y_shape}")

# option: delete memmap views
del X_mm, y_mm

Meta:
  N = 76694
  T = 400
  C = 19
  X_path = Processed\L400\FEPCR\X.dat
  y_path = Processed\L400\FEPCR\y.dat
-------------------------------------
Subject ID 001: X shape=(532, 400, 19), y shape=(532, 3)
Subject ID 002: X shape=(523, 400, 19), y shape=(523, 3)
Subject ID 003: X shape=(523, 400, 19), y shape=(523, 3)
Subject ID 004: X shape=(523, 400, 19), y shape=(523, 3)
Subject ID 005: X shape=(539, 400, 19), y shape=(539, 3)
Subject ID 006: X shape=(523, 400, 19), y shape=(523, 3)
Subject ID 007: X shape=(525, 400, 19), y shape=(525, 3)
Subject ID 008: X shape=(525, 400, 19), y shape=(525, 3)
Subject ID 009: X shape=(543, 400, 19), y shape=(543, 3)
Subject ID 010: X shape=(526, 400, 19), y shape=(526, 3)
Subject ID 011: X shape=(532, 400, 19), y shape=(532, 3)
Subject ID 012: X shape=(529, 400, 19), y shape=(529, 3)
Subject ID 013: X shape=(526, 400, 19), y shape=(526, 3)
Subject ID 014: X shape=(525, 400, 19), y shape=(525, 3)
Subject ID 015: X shape=(525, 400, 19), y shape=(52